# Workshop 8 — Text Classification with Deep Learning
**Saharsh Pathak | 2417371 | Herald College Kathmandu**

Text classification using LSTM, Bidirectional LSTM, and CNN-based models.
Sentiment analysis on movie reviews (IMDB dataset).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

print(f'TensorFlow: {tf.__version__}')
np.random.seed(42)

## 1. Load IMDB Dataset

In [ ]:
VOCAB_SIZE = 10000
MAX_LEN = 200

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print(f'Train: {len(X_train)} reviews')
print(f'Test:  {len(X_test)} reviews')
print(f'Classes: 0=Negative, 1=Positive')
print(f'Class balance: {np.bincount(y_train)}')

# Pad sequences to same length
X_train_pad = pad_sequences(X_train, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test,  maxlen=MAX_LEN, padding='post', truncating='post')
print(f'Padded shape: {X_train_pad.shape}')

## 2. Model 1 — LSTM

In [ ]:
def build_lstm(vocab_size=10000, embed_dim=64, max_len=200):
    """LSTM for sentiment classification."""
    model = keras.Sequential([
        layers.Embedding(vocab_size, embed_dim, input_length=max_len),
        layers.LSTM(64, dropout=0.3, recurrent_dropout=0.3),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ], name='LSTM_Sentiment')
    return model

lstm_model = build_lstm(VOCAB_SIZE, 64, MAX_LEN)
lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm_model.summary()

## 3. Model 2 — Bidirectional LSTM

In [ ]:
def build_bilstm(vocab_size=10000, embed_dim=64, max_len=200):
    """Bidirectional LSTM — reads sequence forward AND backward."""
    model = keras.Sequential([
        layers.Embedding(vocab_size, embed_dim, input_length=max_len),
        layers.Bidirectional(layers.LSTM(64, dropout=0.3)),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid')
    ], name='BiLSTM_Sentiment')
    return model

bilstm_model = build_bilstm(VOCAB_SIZE, 64, MAX_LEN)
bilstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print(f'BiLSTM params: {bilstm_model.count_params():,}')

## 4. Model 3 — 1D CNN for Text

In [ ]:
def build_text_cnn(vocab_size=10000, embed_dim=64, max_len=200):
    """1D CNN for text classification — faster than LSTM."""
    model = keras.Sequential([
        layers.Embedding(vocab_size, embed_dim, input_length=max_len),
        layers.Conv1D(128, 5, activation='relu'),
        layers.GlobalMaxPooling1D(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ], name='TextCNN_Sentiment')
    return model

cnn_model = build_text_cnn(VOCAB_SIZE, 64, MAX_LEN)
cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print(f'TextCNN params: {cnn_model.count_params():,}')

## 5. Train All Models

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2)
]

histories = {}
for name, model in [('LSTM', lstm_model), ('BiLSTM', bilstm_model), ('TextCNN', cnn_model)]:
    print(f'\nTraining {name}...')
    h = model.fit(
        X_train_pad, y_train,
        epochs=10, batch_size=128,
        validation_split=0.1,
        callbacks=callbacks, verbose=0
    )
    histories[name] = h
    val_acc = max(h.history['val_accuracy'])
    print(f'  Best val accuracy: {val_acc:.4f}')

## 6. Results Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'LSTM': '#0EA5E9', 'BiLSTM': '#6366F1', 'TextCNN': '#10B981'}

for name, h in histories.items():
    axes[0].plot(h.history['val_accuracy'], label=name, color=colors[name], linewidth=2)
    axes[1].plot(h.history['val_loss'], label=name, color=colors[name], linewidth=2)

axes[0].set_title('Validation Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title('Validation Loss'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('Text Classification Models Comparison', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Final test accuracy
print('\nFinal Test Accuracy:')
for name, model in [('LSTM', lstm_model), ('BiLSTM', bilstm_model), ('TextCNN', cnn_model)]:
    _, acc = model.evaluate(X_test_pad, y_test, verbose=0)
    print(f'  {name:10s}: {acc:.4f} ({acc*100:.2f}%)')

## 7. Predict on Custom Reviews

In [ ]:
word_index = imdb.get_word_index()

def encode_review(text, word_index, max_len=200):
    tokens = text.lower().split()
    encoded = [word_index.get(w, 2) + 3 for w in tokens]
    return pad_sequences([encoded], maxlen=max_len, padding='post')

reviews = [
    "This movie was absolutely fantastic! Great acting and storyline.",
    "Terrible film. Waste of time. Boring and predictable.",
    "An average movie with some good moments but overall disappointing."
]

print('Sentiment Predictions (BiLSTM):')
print('-' * 60)
for review in reviews:
    enc = encode_review(review, word_index)
    prob = bilstm_model.predict(enc, verbose=0)[0][0]
    sentiment = '😊 POSITIVE' if prob > 0.5 else '😞 NEGATIVE'
    print(f'{sentiment} ({prob:.3f}): {review[:50]}...')